# Module 8: 高级优化技术

## 学习目标
- 理解 CUDA Graph 的原理和应用
- 掌握 Tensor Parallelism 的实现
- 学习 FlashAttention 和 FlashInfer 的集成
- 了解其他性能优化技巧

---

## 8.1 CUDA Graph

### 问题: CPU 启动开销

在 Decode 阶段，每次只处理一个 token，但每次都需要:
1. CPU 调用 CUDA kernel 启动
2. 驱动程序处理
3. GPU 接收命令

这些开销在 Decode 阶段尤其明显。

### 解决方案: CUDA Graph

```
普通执行:
CPU: [Launch K1][Launch K2][Launch K3]...  (每次都有开销)
GPU: [  K1  ][  K2  ][  K3  ]...

CUDA Graph:
1. 捕获阶段: 记录所有操作
2. 重放阶段: 一次调用执行所有操作

CPU: [Replay Graph] (一次调用)
GPU: [K1][K2][K3]...
```

In [ ]:
import torch
import time

# 检查 CUDA 是否可用
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

if device.type == "cuda":
    # 创建一个简单的模型
    class SimpleModel(torch.nn.Module):
        def __init__(self, hidden_size):
            super().__init__()
            self.linear1 = torch.nn.Linear(hidden_size, hidden_size * 4)
            self.linear2 = torch.nn.Linear(hidden_size * 4, hidden_size)
            self.norm = torch.nn.LayerNorm(hidden_size)
        
        def forward(self, x):
            x = self.linear1(x)
            x = torch.relu(x)
            x = self.linear2(x)
            x = self.norm(x)
            return x
    
    hidden_size = 256
    model = SimpleModel(hidden_size).to(device)
    
    # 创建静态输入 (用于 CUDA Graph 捕获)
    static_input = torch.randn(1, hidden_size, device=device)
    static_output = torch.empty(1, hidden_size, device=device)
    
    # 预热
    for _ in range(3):
        _ = model(static_input)
    torch.cuda.synchronize()
    
    # 捕获 CUDA Graph
    stream = torch.cuda.Stream()
    with torch.cuda.stream(stream):
        graph = torch.cuda.CUDAGraph()
        with torch.cuda.graph(graph):
            static_output = model(static_input)
    
    print("CUDA Graph 捕获成功!")
else:
    print("跳过 CUDA Graph 演示 (需要 GPU)")

In [ ]:
if device.type == "cuda":
    # 对比普通执行和 CUDA Graph
    num_iterations = 1000
    
    # 普通执行
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(num_iterations):
        _ = model(static_input)
    torch.cuda.synchronize()
    normal_time = time.time() - start
    
    # CUDA Graph 执行
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(num_iterations):
        graph.replay()
    torch.cuda.synchronize()
    graph_time = time.time() - start
    
    print(f"\n性能对比 ({num_iterations} 次迭代):")
    print(f"  普通执行: {normal_time*1000:.2f} ms")
    print(f"  CUDA Graph: {graph_time*1000:.2f} ms")
    print(f"  加速比: {normal_time/graph_time:.2f}x")

## 8.2 Mini-SGLang 的 CUDA Graph 实现

Mini-SGLang 为不同的 batch size 预先捕获 CUDA Graph。

In [ ]:
from typing import List, Dict, Optional

class GraphRunner:
    """CUDA Graph 运行器 (简化版)"""
    
    def __init__(
        self,
        model,
        cuda_graph_bs: List[int],      # 要捕获的 batch sizes
        cuda_graph_max_bs: int,         # 最大 batch size
    ):
        self.model = model
        self.cuda_graph_bs = cuda_graph_bs
        self.cuda_graph_max_bs = cuda_graph_max_bs
        
        # 存储捕获的 graphs
        self.graphs: Dict[int, torch.cuda.CUDAGraph] = {}
        self.static_inputs: Dict[int, torch.Tensor] = {}
        self.static_outputs: Dict[int, torch.Tensor] = {}
    
    def capture(self, device: torch.device):
        """捕获不同 batch size 的 graphs"""
        print(f"开始捕获 CUDA Graphs...")
        print(f"  Batch sizes: {self.cuda_graph_bs}")
        
        for bs in self.cuda_graph_bs:
            print(f"  捕获 batch_size={bs}...")
            
            # 创建静态缓冲区
            self.static_inputs[bs] = torch.randn(bs, 256, device=device)
            self.static_outputs[bs] = torch.empty(bs, 256, device=device)
            
            # 预热
            for _ in range(3):
                self.model(self.static_inputs[bs])
            torch.cuda.synchronize()
            
            # 捕获
            self.graphs[bs] = torch.cuda.CUDAGraph()
            with torch.cuda.graph(self.graphs[bs]):
                self.static_outputs[bs] = self.model(self.static_inputs[bs])
        
        print("  捕获完成!")
    
    def can_use_cuda_graph(self, batch_size: int) -> bool:
        """检查是否可以使用 CUDA Graph"""
        return batch_size in self.graphs
    
    def pad_batch_size(self, bs: int) -> int:
        """将 batch size 向上取整到已捕获的大小"""
        for captured_bs in sorted(self.cuda_graph_bs):
            if bs <= captured_bs:
                return captured_bs
        return bs  # 超过最大值，使用 eager 执行
    
    def replay(self, batch_size: int, input_data: torch.Tensor) -> torch.Tensor:
        """重放 CUDA Graph"""
        padded_bs = self.pad_batch_size(batch_size)
        
        # 复制输入到静态缓冲区
        self.static_inputs[padded_bs][:batch_size].copy_(input_data)
        
        # 重放 graph
        self.graphs[padded_bs].replay()
        
        # 返回有效部分的输出
        return self.static_outputs[padded_bs][:batch_size]

# 演示
if device.type == "cuda":
    graph_runner = GraphRunner(
        model=model,
        cuda_graph_bs=[1, 2, 4, 8, 16, 32],
        cuda_graph_max_bs=32,
    )
    graph_runner.capture(device)
    
    # 测试
    test_input = torch.randn(5, 256, device=device)
    padded_bs = graph_runner.pad_batch_size(5)
    print(f"\n测试: batch_size=5 被填充到 {padded_bs}")

## 8.3 Tensor Parallelism

当模型太大无法放入单个 GPU 时，使用 Tensor Parallelism 将模型分布到多个 GPU。

In [ ]:
def visualize_tensor_parallelism():
    """可视化 Tensor Parallelism"""
    print("""
    ┌─────────────────────────────────────────────────────────────────┐
    │                    Tensor Parallelism (TP=2)                    │
    ├─────────────────────────────────────────────────────────────────┤
    │                                                                 │
    │  Attention Layer:                                               │
    │  ┌─────────────────────────────────────────────────────────┐   │
    │  │  QKV Linear (Column Parallel)                           │   │
    │  │  ┌─────────────┐     ┌─────────────┐                    │   │
    │  │  │   GPU 0     │     │   GPU 1     │                    │   │
    │  │  │  Q0,K0,V0   │     │  Q1,K1,V1   │                    │   │
    │  │  └──────┬──────┘     └──────┬──────┘                    │   │
    │  │         │                   │                           │   │
    │  │         ▼                   ▼                           │   │
    │  │  ┌─────────────┐     ┌─────────────┐                    │   │
    │  │  │ Attention 0 │     │ Attention 1 │  (独立计算)        │   │
    │  │  └──────┬──────┘     └──────┬──────┘                    │   │
    │  │         │                   │                           │   │
    │  │         ▼                   ▼                           │   │
    │  │  O Projection (Row Parallel)                            │   │
    │  │  ┌─────────────┐     ┌─────────────┐                    │   │
    │  │  │   GPU 0     │     │   GPU 1     │                    │   │
    │  │  └──────┬──────┘     └──────┬──────┘                    │   │
    │  │         │                   │                           │   │
    │  │         └─────► All-Reduce ◄─────┘                      │   │
    │  │                     │                                   │   │
    │  └─────────────────────┼───────────────────────────────────┘   │
    │                        ▼                                       │
    │  MLP Layer:                                                    │
    │  ┌─────────────────────────────────────────────────────────┐   │
    │  │  Gate+Up Linear (Column Parallel)                       │   │
    │  │  ┌─────────────┐     ┌─────────────┐                    │   │
    │  │  │   GPU 0     │     │   GPU 1     │                    │   │
    │  │  │   Gate0     │     │   Gate1     │                    │   │
    │  │  │    Up0      │     │    Up1      │                    │   │
    │  │  └──────┬──────┘     └──────┬──────┘                    │   │
    │  │         │                   │                           │   │
    │  │         ▼                   ▼                           │   │
    │  │  Down Linear (Row Parallel)                             │   │
    │  │  ┌─────────────┐     ┌─────────────┐                    │   │
    │  │  │   GPU 0     │     │   GPU 1     │                    │   │
    │  │  └──────┬──────┘     └──────┬──────┘                    │   │
    │  │         │                   │                           │   │
    │  │         └─────► All-Reduce ◄─────┘                      │   │
    │  └─────────────────────────────────────────────────────────┘   │
    │                                                                 │
    └─────────────────────────────────────────────────────────────────┘
    """)

visualize_tensor_parallelism()

In [ ]:
# 模拟 Tensor Parallelism 的通信
def simulate_tp_communication():
    """模拟 TP 通信开销"""
    print("Tensor Parallelism 通信分析:")
    print("="*60)
    
    # 假设的模型配置
    hidden_size = 4096
    num_layers = 32
    batch_size = 1
    seq_len = 1  # decode 阶段
    dtype_bytes = 2  # bfloat16
    
    # 每层需要 2 次 all-reduce:
    # 1. Attention O projection 后
    # 2. MLP down projection 后
    all_reduce_per_layer = 2
    
    # 每次 all-reduce 的数据量
    data_per_allreduce = batch_size * seq_len * hidden_size * dtype_bytes
    
    # 总通信量
    total_comm = all_reduce_per_layer * num_layers * data_per_allreduce
    
    print(f"\n模型配置:")
    print(f"  hidden_size: {hidden_size}")
    print(f"  num_layers: {num_layers}")
    print(f"  dtype: bfloat16")
    
    print(f"\n通信分析 (每个 decode step):")
    print(f"  每层 all-reduce 次数: {all_reduce_per_layer}")
    print(f"  每次 all-reduce 数据量: {data_per_allreduce} bytes")
    print(f"  总通信量: {total_comm / 1024:.2f} KB")
    
    # NVLink 带宽估算
    nvlink_bw = 600 * 1024 * 1024 * 1024  # 600 GB/s (NVLink 4.0)
    estimated_time = total_comm / nvlink_bw * 1000 * 1000  # us
    print(f"\n通信延迟估算 (NVLink 600GB/s):")
    print(f"  理论最小延迟: {estimated_time:.2f} μs")

simulate_tp_communication()

## 8.4 FlashAttention 和 FlashInfer

### FlashAttention
- 用于 **Prefill 阶段**
- 优化长序列的密集 attention
- 减少 HBM 访问

### FlashInfer
- 用于 **Decode 阶段**
- 支持 Paged KV Cache
- 针对单 token 生成优化

In [ ]:
def compare_attention_backends():
    """对比不同 Attention 后端"""
    print("""
    ┌─────────────────────────────────────────────────────────────────┐
    │              Attention Backend 对比                            │
    ├─────────────────┬───────────────────┬───────────────────────────┤
    │     特性        │  FlashAttention3  │      FlashInfer          │
    ├─────────────────┼───────────────────┼───────────────────────────┤
    │  适用阶段       │  Prefill          │  Decode                   │
    │  序列长度       │  长序列 (批量)    │  单 token                 │
    │  KV Cache       │  连续存储         │  分页存储 (Paged)         │
    │  内存效率       │  O(n) 峰值内存    │  动态分配                 │
    │  计算优化       │  分块 + SRAM      │  优化的 decode kernel     │
    │  CUDA Graph     │  不常用           │  完全支持                 │
    └─────────────────┴───────────────────┴───────────────────────────┘
    """)

compare_attention_backends()

## 8.5 其他优化技巧

### 1. 内存优化
- **Activation Checkpointing**: 减少激活内存
- **Mixed Precision**: BFloat16 节省内存
- **KV Cache 量化**: 进一步压缩

### 2. 计算优化
- **Fused Kernels**: 减少内存访问
- **Speculative Decoding**: 加速生成

### 3. 调度优化
- **Smart Batching**: 根据长度分组
- **Priority Scheduling**: 优先处理短请求

In [ ]:
def optimization_checklist():
    """优化检查清单"""
    optimizations = [
        ("CUDA Graph", "Decode 阶段", "减少 CPU 启动开销"),
        ("FlashAttention", "Prefill 阶段", "减少 HBM 访问"),
        ("FlashInfer", "Decode 阶段", "优化 Paged Attention"),
        ("Tensor Parallelism", "大模型", "分布到多 GPU"),
        ("Radix Cache", "多请求", "复用相同前缀"),
        ("Chunked Prefill", "长输入", "减少峰值内存"),
        ("Overlap Scheduling", "全阶段", "隐藏 CPU 开销"),
        ("Continuous Batching", "在线服务", "提高 GPU 利用率"),
    ]
    
    print("Mini-SGLang 优化技术清单:")
    print("="*70)
    print(f"{'技术':<20} {'适用场景':<15} {'效果':<35}")
    print("-"*70)
    for tech, scenario, effect in optimizations:
        print(f"{tech:<20} {scenario:<15} {effect:<35}")

optimization_checklist()

## 8.6 性能调优建议

### 1. 根据场景选择配置

```bash
# 高吞吐场景 (离线批处理)
python -m minisgl --model "Qwen/Qwen3-32B" --tp 4 \
    --max-running-req 512 \
    --cuda-graph-max-bs 64

# 低延迟场景 (在线服务)
python -m minisgl --model "Qwen/Qwen3-32B" --tp 4 \
    --max-running-req 128 \
    --cuda-graph-max-bs 32

# 长上下文场景
python -m minisgl --model "Qwen/Qwen3-32B" --tp 4 \
    --max-prefill-length 8192 \
    --max-seq-len 131072
```

### 2. 监控关键指标
- **GPU 利用率**: 应该 > 80%
- **内存使用**: 接近但不超过 GPU 内存
- **批大小分布**: 检查是否充分利用了批处理

## 8.7 小结

### 核心优化技术:

1. **CUDA Graph**:
   - 预先捕获 kernel 序列
   - 消除 CPU 启动开销
   - 特别适合 Decode 阶段

2. **Tensor Parallelism**:
   - Column Parallel: QKV, Gate+Up
   - Row Parallel: O Proj, Down
   - 需要 All-Reduce 通信

3. **FlashAttention/FlashInfer**:
   - FlashAttention: Prefill 阶段
   - FlashInfer: Decode + Paged KV

4. **其他优化**:
   - Radix Cache
   - Chunked Prefill
   - Overlap Scheduling
   - Continuous Batching

---

恭喜你完成了 Mini-SGLang 学习指南的全部内容！

**下一步建议**:
1. 阅读源代码，特别是 `scheduler.py` 和 `engine.py`
2. 尝试修改和扩展代码
3. 运行 benchmark 进行性能测试
4. 贡献到 SGLang 社区!